## Traficom Summary Merge Preparation
This notebook prepares the price data and cleaned Traficom summary tables for merge by loading the cleaned summary files, checking merge-key quality, repairing merge keys in `price_df`, and validating that the datasets are ready for a later merge.

## Step 1 — Load datasets

In [103]:
from pathlib import Path
from IPython.display import display
import csv
import pandas as pd

candidate_roots = [Path.cwd(), *Path.cwd().parents]
base_dir = next(
    (
        path
        for path in candidate_roots
        if (path / "datasets").exists() and (path / "notebooks").exists()
    ),
    None,
)

if base_dir is None:
    raise FileNotFoundError("Could not locate the project root from the current notebook working directory.")

price_path = base_dir / "datasets/cleaned/cleaned_and_merged_pricedataset.csv"
with price_path.open("r", encoding="utf-8", errors="replace", newline="") as f:
    reader = csv.reader(f, skipinitialspace=True)
    price_columns = [column.strip() for column in next(reader)]
    price_rows = list(reader)

price_df = pd.DataFrame(price_rows, columns=price_columns)
brand_summary_df = pd.read_csv(base_dir / "datasets/traficom_outputs/brand_summary.csv")
model_summary_df = pd.read_csv(base_dir / "datasets/traficom_outputs/model_summary.csv")

for column in price_df.columns:
    price_df[column] = price_df[column].astype("string").str.strip()

for column in ["product_id", "price", "mileage", "year_start", "year_end", "year_span", "year_mid"]:
    if column in price_df.columns:
        price_df[column] = pd.to_numeric(price_df[column], errors="coerce")


## Step 2 — Basic inspection

In [104]:
print(f"price_df shape: {price_df.shape}")
print(f"brand_summary_df shape: {brand_summary_df.shape}")
print(f"model_summary_df shape: {model_summary_df.shape}")

print("\nRelevant price_df columns:")
print([col for col in ["product_id", "part_name", "brand", "model", "category", "subcategory"] if col in price_df.columns])

print("\nRelevant brand_summary_df columns:")
print([col for col in ["brand", "brand_total_registered"] if col in brand_summary_df.columns])

print("\nRelevant model_summary_df columns:")
print([col for col in ["brand", "model_family_clean", "model_total_registered"] if col in model_summary_df.columns])

price_df shape: (11374, 15)
brand_summary_df shape: (767, 18)
model_summary_df shape: (18650, 20)

Relevant price_df columns:
['product_id']

Relevant brand_summary_df columns:
['brand', 'brand_total_registered']

Relevant model_summary_df columns:
['brand', 'model_family_clean', 'model_total_registered']


## Step 3 — Standardize merge-key text

In [105]:
def standardize_key(series):
    series = series.astype("string").str.strip().str.lower()
    return series.replace({"": pd.NA, "nan": pd.NA})

for df in [price_df, brand_summary_df, model_summary_df]:
    df.columns = df.columns.str.strip()

price_df["brand"] = standardize_key(price_df["brand"])
price_df["model"] = standardize_key(price_df["model"])
price_df["category"] = standardize_key(price_df["category"])
price_df["subcategory"] = standardize_key(price_df["subcategory"])

brand_summary_df["brand"] = standardize_key(brand_summary_df["brand"])

model_summary_df["brand"] = standardize_key(model_summary_df["brand"])
model_summary_df["model_family_clean"] = standardize_key(model_summary_df["model_family_clean"])

## Step 4 — Check merge-key completeness and uniqueness

In [106]:
missing_key_summary_df = pd.DataFrame(
    {
        "dataset": ["price_df", "model_summary_df", "brand_summary_df"],
        "missing_brand": [
            int(price_df["brand"].isna().sum()),
            int(model_summary_df["brand"].isna().sum()),
            int(brand_summary_df["brand"].isna().sum()),
        ],
        "missing_model": [
            int(price_df["model"].isna().sum()),
            int(model_summary_df["model_family_clean"].isna().sum()),
            pd.NA,
        ],
    }
)
display(missing_key_summary_df)

print("Duplicate brand keys in brand_summary_df:", int(brand_summary_df[["brand"]].dropna().duplicated().sum()))
print(
    "Duplicate brand-model keys in model_summary_df:",
    int(model_summary_df[["brand", "model_family_clean"]].dropna().duplicated().sum()),
)

,dataset,missing_brand,missing_model
0,price_df,0,0
1,model_summary_df,15,497
2,brand_summary_df,1,<NA>


Duplicate brand keys in brand_summary_df: 0
Duplicate brand-model keys in model_summary_df: 0


## Step 5 — Diagnose `price_df` merge-key consistency

In [107]:
known_manufacturer_set = set(brand_summary_df["brand"].dropna().unique())
known_model_family_set = set(model_summary_df["model_family_clean"].dropna().unique())

price_df["category_key"] = price_df["category"]
price_df["subcategory_key"] = price_df["subcategory"]

part_taxonomy_terms = set(price_df["category_key"].dropna()) | set(price_df["subcategory_key"].dropna())
part_keyword_pattern = r"engine|brake|fuel|airbag|suspension|gear ?box|axle|electric|databox|sensor|transmitter|vehicle exterior|drive axle|middle axle"

price_df["model_looks_like_part_taxonomy"] = (
    price_df["model"].eq(price_df["category_key"])
    | price_df["model"].eq(price_df["subcategory_key"])
    | price_df["model"].isin(part_taxonomy_terms)
    | price_df["model"].astype("string").str.contains(part_keyword_pattern, case=False, na=False, regex=True)
)
price_df["brand_is_known_model_family"] = price_df["brand"].isin(known_model_family_set)

diagnostic_summary_df = pd.DataFrame(
    {
        "diagnostic": [
            "brand is known manufacturer",
            "brand is known model family",
            "model is known model family",
            "model looks like part taxonomy",
        ],
        "row_count": [
            int(price_df["brand"].isin(known_manufacturer_set).sum()),
            int(price_df["brand_is_known_model_family"].sum()),
            int(price_df["model"].isin(known_model_family_set).sum()),
            int(price_df["model_looks_like_part_taxonomy"].sum()),
        ],
    }
)
diagnostic_summary_df["share_of_rows"] = diagnostic_summary_df["row_count"] / len(price_df)
display(diagnostic_summary_df)

suspicious_pairs_df = (
    price_df.loc[
        price_df["brand_is_known_model_family"] | price_df["model_looks_like_part_taxonomy"],
        ["brand", "model"],
    ]
    .value_counts()
    .rename("row_count")
    .reset_index()
)
display(suspicious_pairs_df.head(10))

,diagnostic,row_count,share_of_rows
0,brand is known manufacturer,0,0.000000
1,brand is known model family,11374,1.000000
2,model is known model family,3790,0.333216
3,model looks like part taxonomy,7584,0.666784


,brand,model,row_count
0,vw,golf,3790
1,octavia,electric / transmitter / databox / sensor,1044
2,corolla,electric / transmitter / databox / sensor,872
3,corolla,vehicle exterior / suspension,801
4,corolla,engine,572
5,octavia,engine,540
6,corolla,brakes,512
7,octavia,gear box / drive axle / middle axle,498
8,octavia,vehicle exterior / suspension,486
9,corolla,fuel,472


## Step 6 — Repair merge keys safely

In [108]:
manufacturer_alias_map = {"vw": "volkswagen"}

model_family_brand_counts = (
    model_summary_df[["brand", "model_family_clean"]]
    .dropna()
    .drop_duplicates()
    .groupby("model_family_clean")["brand"]
    .nunique()
)
unique_model_brand_map = (
    model_summary_df[["brand", "model_family_clean"]]
    .dropna()
    .drop_duplicates()
    .groupby("model_family_clean")["brand"]
    .first()
    .loc[model_family_brand_counts.eq(1)]
)
ambiguous_model_family_set = set(model_family_brand_counts[model_family_brand_counts.gt(1)].index)

traficom_pair_index = pd.MultiIndex.from_frame(
    model_summary_df[["brand", "model_family_clean"]].dropna().drop_duplicates()
)

price_df["brand_canonical_candidate"] = price_df["brand"].replace(manufacturer_alias_map)
price_df["brand_merge_key"] = pd.NA
price_df["model_merge_key"] = pd.NA
price_df["repair_status"] = "unknown_brand_model"

original_valid_mask = pd.MultiIndex.from_arrays(
    [price_df["brand_canonical_candidate"], price_df["model"]]
).isin(traficom_pair_index)

price_df.loc[original_valid_mask, "brand_merge_key"] = price_df.loc[original_valid_mask, "brand_canonical_candidate"]
price_df.loc[original_valid_mask, "model_merge_key"] = price_df.loc[original_valid_mask, "model"]
price_df.loc[original_valid_mask, "repair_status"] = "original_valid"

repair_from_model_family_mask = (
    ~original_valid_mask
    & price_df["brand"].isin(unique_model_brand_map.index)
    & price_df["model_looks_like_part_taxonomy"]
)
price_df.loc[repair_from_model_family_mask, "brand_merge_key"] = price_df.loc[
    repair_from_model_family_mask, "brand"
].map(unique_model_brand_map)
price_df.loc[repair_from_model_family_mask, "model_merge_key"] = price_df.loc[
    repair_from_model_family_mask, "brand"
]
price_df.loc[repair_from_model_family_mask, "repair_status"] = "repaired_from_known_model_family"

ambiguous_mask = (
    price_df["repair_status"].eq("unknown_brand_model")
    & price_df["brand"].isin(ambiguous_model_family_set)
    & price_df["model_looks_like_part_taxonomy"]
)
price_df.loc[ambiguous_mask, "repair_status"] = "ambiguous_unrepaired"

invalid_taxonomy_mask = (
    price_df["repair_status"].eq("unknown_brand_model")
    & price_df["model_looks_like_part_taxonomy"]
)
price_df.loc[invalid_taxonomy_mask, "repair_status"] = "invalid_part_taxonomy_in_model"

## Step 7 — Validate repaired merge keys

In [109]:
original_pair_index = pd.MultiIndex.from_frame(price_df[["brand", "model"]])
repaired_pair_index = pd.MultiIndex.from_frame(price_df[["brand_merge_key", "model_merge_key"]])

matched_row_share_before = original_pair_index.isin(traficom_pair_index).mean()
matched_row_share_after = repaired_pair_index.isin(traficom_pair_index).mean()

comparison_summary_df = pd.DataFrame(
    {
        "metric": [
            "matched row share",
            "non-missing repaired key rows",
        ],
        "before": [round(float(matched_row_share_before), 4), int(price_df[["brand", "model"]].dropna().shape[0])],
        "after": [round(float(matched_row_share_after), 4), int(price_df[["brand_merge_key", "model_merge_key"]].dropna().shape[0])],
    }
)
display(comparison_summary_df)

repair_status_summary_df = (
    price_df["repair_status"]
    .value_counts(dropna=False)
    .rename_axis("repair_status")
    .reset_index(name="row_count")
)
display(repair_status_summary_df)

top_repaired_pairs_df = (
    price_df[["brand_merge_key", "model_merge_key"]]
    .dropna()
    .value_counts()
    .rename("row_count")
    .reset_index()
)
display(top_repaired_pairs_df.head(10))

,metric,before,after
0,matched row share,0.0,1.0
1,non-missing repaired key rows,11374.0,11374.0


,repair_status,row_count
0,repaired_from_known_model_family,7584
1,original_valid,3790


,brand_merge_key,model_merge_key,row_count
0,toyota,corolla,3947
1,volkswagen,golf,3790
2,skoda,octavia,3637


## Step 8 — Final merge-readiness check

In [110]:
brand_key_duplicates = int(brand_summary_df[["brand"]].dropna().duplicated().sum())
model_key_duplicates = int(
    model_summary_df[["brand", "model_family_clean"]].dropna().duplicated().sum()
)

print("Duplicate brand keys in brand_summary_df:", brand_key_duplicates)
print("Duplicate brand-model keys in model_summary_df:", model_key_duplicates)
print("Recommended validate setting:", "many_to_one" if model_key_duplicates == 0 else "not safe for many_to_one")

test_merge_df = price_df[[col for col in ["product_id", "brand_merge_key", "model_merge_key", "repair_status"] if col in price_df.columns]].copy()
test_merge_df = test_merge_df.dropna(subset=["brand_merge_key", "model_merge_key"])

test_merge_result_df = test_merge_df.merge(
    model_summary_df[["brand", "model_family_clean"]].drop_duplicates(),
    how="left",
    left_on=["brand_merge_key", "model_merge_key"],
    right_on=["brand", "model_family_clean"],
    indicator=True,
    validate="many_to_one",
)

print("\nTest merge indicator counts")
print(test_merge_result_df["_merge"].value_counts(dropna=False).to_string())

Duplicate brand keys in brand_summary_df: 0
Duplicate brand-model keys in model_summary_df: 0
Recommended validate setting: many_to_one

Test merge indicator counts
_merge
both          11374
left_only         0
right_only        0


## Step 9 — Short conclusion
- The cleaned Traficom summary CSVs now load normally and are the files used in this notebook.
- The Traficom summary tables are unique on their intended merge keys.
- The original `price_df` merge keys were not merge-ready because model-family and part-taxonomy text were mixed into the raw fields.
- Conservative repaired merge keys were created in `brand_merge_key` and `model_merge_key` while preserving the original columns.
- The notebook is now ready for the actual merge step in a separate notebook or next section, but it does not perform the final permanent merge yet.

## Step 10 — Merge price_df with model summary


In [111]:
merged_df_model = price_df.merge(
    model_summary_df,
    how="left",
    left_on=["brand_merge_key", "model_merge_key"],
    right_on=["brand", "model_family_clean"],
    validate="many_to_one",
    indicator=True,
)

print(f"price_df shape before model merge: {price_df.shape}")
print(f"merged_df_model shape after model merge: {merged_df_model.shape}")
print("\nModel merge indicator counts")
print(merged_df_model["_merge"].value_counts(dropna=False).to_string())

model_merge_brand_column = "brand_x" if "brand_x" in merged_df_model.columns else "brand"

model_left_only_count = int(merged_df_model["_merge"].eq("left_only").sum())
print(f"\nModel merge left_only rows: {model_left_only_count}")
if model_left_only_count > 0:
    display(
        merged_df_model.loc[
            merged_df_model["_merge"].eq("left_only"),
            [col for col in ["product_id", model_merge_brand_column, "model", "brand_merge_key", "model_merge_key", "repair_status"] if col in merged_df_model.columns],
        ].head(20)
    )
    raise ValueError("Unexpected left_only rows found in the model merge. Review the repaired merge keys before proceeding.")

display(
    merged_df_model[
        [
            col
            for col in [
                "product_id",
                "part_name",
                model_merge_brand_column,
                "model",
                "brand_merge_key",
                "model_merge_key",
                "repair_status",
                "model_total_registered",
                "model_share_within_brand",
                "model_share_of_market",
            ]
            if col in merged_df_model.columns
        ]
    ].head(10)
)


price_df shape before model merge: (11374, 23)
merged_df_model shape after model merge: (11374, 44)

Model merge indicator counts
_merge
both          11374
left_only         0
right_only        0

Model merge left_only rows: 0


,product_id,part_name,brand_x,model,brand_merge_key,model_merge_key,repair_status,model_total_registered,model_share_within_brand,model_share_of_market
0,"""Contact roll Airbag -","e-""",vw,golf,volkswagen,golf,original_valid,86762,0.310629,0.033825
1,"""Contact roll Airbag -","e-""",vw,golf,volkswagen,golf,original_valid,86762,0.310629,0.033825
2,"""Contact roll Airbag -","e-""",vw,golf,volkswagen,golf,original_valid,86762,0.310629,0.033825
3,"""Contact roll Airbag -","e-""",vw,golf,volkswagen,golf,original_valid,86762,0.310629,0.033825
4,"""Contact roll Airbag -","e-""",vw,golf,volkswagen,golf,original_valid,86762,0.310629,0.033825
5,"""Other airbags -","e-""",vw,golf,volkswagen,golf,original_valid,86762,0.310629,0.033825
6,"""Other airbags -","e-""",vw,golf,volkswagen,golf,original_valid,86762,0.310629,0.033825
7,"""Other airbags -","e-""",vw,golf,volkswagen,golf,original_valid,86762,0.310629,0.033825
8,"""Other airbags -","e-""",vw,golf,volkswagen,golf,original_valid,86762,0.310629,0.033825
9,"""Passenger airbag -","e-""",vw,golf,volkswagen,golf,original_valid,86762,0.310629,0.033825


## Step 11 — Clean model-merge result columns


In [112]:
merged_df_model_clean = merged_df_model.drop(columns=["_merge"]).rename(
    columns={"brand_x": "brand", "brand_y": "traficom_model_brand"}
)

model_helper_columns_to_drop = [
    "category_key",
    "subcategory_key",
    "brand_canonical_candidate",
    "traficom_model_brand",
]
merged_df_model_clean = merged_df_model_clean.drop(
    columns=[col for col in model_helper_columns_to_drop if col in merged_df_model_clean.columns]
)

model_feature_exclusions = {"model_merge_key", "model_family_clean", "model_looks_like_part_taxonomy"}
model_feature_columns = [
    col for col in merged_df_model_clean.columns if col.startswith("model_") and col not in model_feature_exclusions
]
price_core_columns = [
    col
    for col in [
        "product_id",
        "part_name",
        "price",
        "quality_grade",
        "oem_number",
        "mileage",
        "brand",
        "model",
        "category",
        "subcategory",
        "scrape_date",
        "year_start",
        "year_end",
        "year_span",
        "year_mid",
        "brand_merge_key",
        "model_merge_key",
        "repair_status",
        "brand_is_known_model_family",
        "model_looks_like_part_taxonomy",
    ]
    if col in merged_df_model_clean.columns
]
remaining_model_columns = [
    col for col in merged_df_model_clean.columns if col not in price_core_columns + model_feature_columns
]
merged_df_model_clean = merged_df_model_clean[price_core_columns + remaining_model_columns + model_feature_columns]

print(f"merged_df_model_clean shape: {merged_df_model_clean.shape}")
print("Columns retained after model-merge cleanup:")
print(merged_df_model_clean.columns.tolist())


merged_df_model_clean shape: (11374, 41)
Columns retained after model-merge cleanup:
['product_id', 'part_name', 'price', 'quality_grade', 'oem_number', 'mileage', 'brand', 'model', 'category', 'subcategory', 'scrape_date', 'year_start', 'year_end', 'year_span', 'year_mid', 'brand_merge_key', 'model_merge_key', 'repair_status', 'brand_is_known_model_family', 'model_looks_like_part_taxonomy', 'model_looks_like_part_taxonomy', 'model_merge_key', 'model_family_clean', 'model_total_registered', 'model_median_vehicle_age', 'model_mean_vehicle_age', 'model_median_mileage', 'model_mean_mileage', 'model_median_engine_cc', 'model_median_power_kw', 'model_median_mass_kg', 'model_share_of_market', 'model_share_within_brand', 'model_share_over_10y', 'model_share_over_200k_km', 'model_automatic_share', 'model_petrol_share', 'model_diesel_share', 'model_ev_share', 'model_hybrid_share', 'model_turbo_share']


## Step 12 — Merge with brand summary


In [113]:
merged_df_final = merged_df_model_clean.merge(
    brand_summary_df,
    how="left",
    left_on="brand_merge_key",
    right_on="brand",
    validate="many_to_one",
    indicator=True,
)

print(f"merged_df_model_clean shape before brand merge: {merged_df_model_clean.shape}")
print(f"merged_df_final shape after brand merge: {merged_df_final.shape}")
print("\nBrand merge indicator counts")
print(merged_df_final["_merge"].value_counts(dropna=False).to_string())

brand_merge_display_column = "brand_x" if "brand_x" in merged_df_final.columns else "brand"

brand_left_only_count = int(merged_df_final["_merge"].eq("left_only").sum())
print(f"\nBrand merge left_only rows: {brand_left_only_count}")
if brand_left_only_count > 0:
    display(
        merged_df_final.loc[
            merged_df_final["_merge"].eq("left_only"),
            [col for col in ["product_id", brand_merge_display_column, "brand_merge_key", "repair_status"] if col in merged_df_final.columns],
        ].head(20)
    )
    raise ValueError("Unexpected left_only rows found in the brand merge. Review the repaired brand keys before proceeding.")

display(
    merged_df_final[
        [
            col
            for col in [
                "product_id",
                "part_name",
                brand_merge_display_column,
                "model",
                "brand_merge_key",
                "model_merge_key",
                "repair_status",
                "brand_total_registered",
                "brand_share_of_market",
            ]
            if col in merged_df_final.columns
        ]
    ].head(10)
)


merged_df_model_clean shape before brand merge: (11374, 41)
merged_df_final shape after brand merge: (11374, 60)

Brand merge indicator counts
_merge
both          11374
left_only         0
right_only        0

Brand merge left_only rows: 0


,product_id,part_name,brand_x,model,brand_merge_key,model_merge_key,model_merge_key,repair_status,brand_total_registered,brand_share_of_market
0,"""Contact roll Airbag -","e-""",vw,golf,volkswagen,golf,golf,original_valid,279311,0.108891
1,"""Contact roll Airbag -","e-""",vw,golf,volkswagen,golf,golf,original_valid,279311,0.108891
2,"""Contact roll Airbag -","e-""",vw,golf,volkswagen,golf,golf,original_valid,279311,0.108891
3,"""Contact roll Airbag -","e-""",vw,golf,volkswagen,golf,golf,original_valid,279311,0.108891
4,"""Contact roll Airbag -","e-""",vw,golf,volkswagen,golf,golf,original_valid,279311,0.108891
5,"""Other airbags -","e-""",vw,golf,volkswagen,golf,golf,original_valid,279311,0.108891
6,"""Other airbags -","e-""",vw,golf,volkswagen,golf,golf,original_valid,279311,0.108891
7,"""Other airbags -","e-""",vw,golf,volkswagen,golf,golf,original_valid,279311,0.108891
8,"""Other airbags -","e-""",vw,golf,volkswagen,golf,golf,original_valid,279311,0.108891
9,"""Passenger airbag -","e-""",vw,golf,volkswagen,golf,golf,original_valid,279311,0.108891


## Step 13 — Clean final merged dataset


In [114]:
merged_df_final = merged_df_final.drop(columns=["_merge"]).rename(columns={"brand_x": "brand", "brand_y": "traficom_brand"})

final_helper_columns_to_drop = ["traficom_brand"]
merged_df_final = merged_df_final.drop(columns=[col for col in final_helper_columns_to_drop if col in merged_df_final.columns])

model_feature_exclusions = {"model_merge_key", "model_family_clean", "model_looks_like_part_taxonomy"}
model_feature_columns = [
    col for col in merged_df_final.columns if col.startswith("model_") and col not in model_feature_exclusions
]
brand_feature_columns = [col for col in merged_df_final.columns if col.startswith("brand_") and col not in {"brand_merge_key", "brand_is_known_model_family"}]
price_columns = [
    col
    for col in [
        "product_id",
        "part_name",
        "price",
        "quality_grade",
        "oem_number",
        "mileage",
        "brand",
        "model",
        "category",
        "subcategory",
        "scrape_date",
        "year_start",
        "year_end",
        "year_span",
        "year_mid",
    ]
    if col in merged_df_final.columns
]
merge_key_columns = [
    col
    for col in [
        "brand_merge_key",
        "model_merge_key",
        "repair_status",
        "brand_is_known_model_family",
        "model_looks_like_part_taxonomy",
        "model_family_clean",
    ]
    if col in merged_df_final.columns
]
other_columns = [
    col
    for col in merged_df_final.columns
    if col not in price_columns + merge_key_columns + model_feature_columns + brand_feature_columns
]
merged_df_final = merged_df_final[price_columns + merge_key_columns + other_columns + model_feature_columns + brand_feature_columns]

print(f"Final merged dataset shape after cleanup: {merged_df_final.shape}")
display(merged_df_final.head(10))


Final merged dataset shape after cleanup: (11374, 67)


,product_id,part_name,price,quality_grade,oem_number,mileage,brand,model,category,subcategory,...,brand_median_mass_kg,brand_share_of_market,brand_share_over_10y,brand_share_over_200k_km,brand_automatic_share,brand_petrol_share,brand_diesel_share,brand_ev_share,brand_hybrid_share,brand_turbo_share
0,"""Contact roll Airbag -","e-""",177.6,A2,FI02042722A,224000.0,vw,golf,airbag,contact roll airbag,...,1400.0,0.108891,0.53811,0.386669,0.478356,0.656791,0.246449,0.074727,0.073083,0.686131
1,"""Contact roll Airbag -","e-""",177.6,A2,FI05351686A,272000.0,vw,golf,airbag,contact roll airbag,...,1400.0,0.108891,0.53811,0.386669,0.478356,0.656791,0.246449,0.074727,0.073083,0.686131
2,"""Contact roll Airbag -","e-""",177.6,A2,FI27837687A,134000.0,vw,golf,airbag,contact roll airbag,...,1400.0,0.108891,0.53811,0.386669,0.478356,0.656791,0.246449,0.074727,0.073083,0.686131
3,"""Contact roll Airbag -","e-""",177.6,A2,FI27837687A,253000.0,vw,golf,airbag,contact roll airbag,...,1400.0,0.108891,0.53811,0.386669,0.478356,0.656791,0.246449,0.074727,0.073083,0.686131
4,"""Contact roll Airbag -","e-""",177.6,A2,FI09389104A,152000.0,vw,golf,airbag,contact roll airbag,...,1400.0,0.108891,0.53811,0.386669,0.478356,0.656791,0.246449,0.074727,0.073083,0.686131
5,"""Other airbags -","e-""",23.7,A2,FI03986645A,,vw,golf,airbag,other airbags,...,1400.0,0.108891,0.53811,0.386669,0.478356,0.656791,0.246449,0.074727,0.073083,0.686131
6,"""Other airbags -","e-""",23.7,A2,FI10331575A,84000.0,vw,golf,airbag,other airbags,...,1400.0,0.108891,0.53811,0.386669,0.478356,0.656791,0.246449,0.074727,0.073083,0.686131
7,"""Other airbags -","e-""",17.8,A2,FI27837687A,227000.0,vw,golf,airbag,other airbags,...,1400.0,0.108891,0.53811,0.386669,0.478356,0.656791,0.246449,0.074727,0.073083,0.686131
8,"""Other airbags -","e-""",14.2,A2,FI06408963A,105000.0,vw,golf,airbag,other airbags,...,1400.0,0.108891,0.53811,0.386669,0.478356,0.656791,0.246449,0.074727,0.073083,0.686131
9,"""Passenger airbag -","e-""",473.6,A1,FI10331575A,34000.0,vw,golf,airbag,passenger airbag,...,1400.0,0.108891,0.53811,0.386669,0.478356,0.656791,0.246449,0.074727,0.073083,0.686131


## Step 14 — Final validation checks


In [115]:
required_objects = ["price_df", "merged_df_final"]
missing_objects = [name for name in required_objects if name not in globals()]
if missing_objects:
    raise RuntimeError(
        "Step 14 depends on earlier merge cells. Rerun the notebook from Step 1 through Step 13 after the load and merge fixes. "
        f"Missing objects: {missing_objects}"
    )

final_row_count = len(merged_df_final)
final_column_count = merged_df_final.shape[1]
row_count_matches_price_df = final_row_count == len(price_df)
row_count_difference = final_row_count - len(price_df)

print(f"Final dataset shape: {merged_df_final.shape}")
print(f"Final row count: {final_row_count}")
print(f"Final column count: {final_column_count}")
print(f"Row count matches price_df: {row_count_matches_price_df}")
print(f"Row count difference vs price_df: {row_count_difference}")

if not row_count_matches_price_df:
    raise ValueError("The final merge changed the row count unexpectedly. Review duplicate keys before using this dataset.")

missing_merge_features_df = pd.DataFrame(
    {
        "column": [
            "brand_merge_key",
            "model_merge_key",
            "model_total_registered",
            "brand_total_registered",
        ],
        "missing_rows": [
            int(merged_df_final["brand_merge_key"].isna().sum()),
            int(merged_df_final["model_merge_key"].isna().sum()),
            int(merged_df_final["model_total_registered"].isna().sum()),
            int(merged_df_final["brand_total_registered"].isna().sum()),
        ],
    }
)
display(missing_merge_features_df)

repair_status_counts_df = (
    merged_df_final["repair_status"]
    .value_counts(dropna=False)
    .rename_axis("repair_status")
    .reset_index(name="row_count")
)
display(repair_status_counts_df)

feature_coverage_df = pd.DataFrame(
    {
        "feature_group": ["model-level features", "brand-level features"],
        "all_rows_covered": [
            bool(merged_df_final["model_total_registered"].notna().all()),
            bool(merged_df_final["brand_total_registered"].notna().all()),
        ],
        "missing_rows": [
            int(merged_df_final["model_total_registered"].isna().sum()),
            int(merged_df_final["brand_total_registered"].isna().sum()),
        ],
    }
)
display(feature_coverage_df)

sample_columns = [
    col
    for col in [
        "product_id",
        "part_name",
        "brand",
        "model",
        "brand_merge_key",
        "model_merge_key",
        "repair_status",
        "model_total_registered",
        "brand_total_registered",
    ]
    if col in merged_df_final.columns
]
display(merged_df_final[sample_columns].head(10))


Final dataset shape: (11374, 67)
Final row count: 11374
Final column count: 67
Row count matches price_df: True
Row count difference vs price_df: 0


TypeError: cannot convert the series to <class 'int'>

## Step 15 — Save final merged dataset


In [ ]:
output_dir = base_dir / "datasets/merged"
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / "price_traficom_merged.csv"

merged_df_final.to_csv(output_path, index=False)
print(f"Saved merged dataset to: {output_path}")

reloaded_merged_df = pd.read_csv(output_path)
print(f"Reloaded merged dataset shape: {reloaded_merged_df.shape}")


## Step 16 — Short final notebook conclusion
- Merge preparation was completed earlier in this notebook using cleaned Traficom summary tables and repaired price-data keys.
- The actual merge has now been performed using `brand_merge_key` and `model_merge_key`, not the raw source fields.
- Row count was preserved through both merge steps, and the merged dataset was saved successfully.
- The combined dataset is now ready for downstream analysis and modeling.
